<div style="padding: 20px; background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); border-radius: 12px; color: white; margin-bottom: 20px; box-shadow: 0 4px 15px rgba(0,0,0,0.2);">
    <h1 style="color: white; margin-bottom: 5px; font-family: 'Segoe UI', sans-serif; font-weight: 700;">🚦 TRAFFIC DENSITY ANALYSIS SYSTEM</h1>
    <p style="font-size: 1.15em; opacity: 0.95; margin-bottom: 0;">Hệ Thống Phân Tích Mật Độ Giao Thông</p>
</div>

### Chào mừng bạn đến với Bảng Điều Khiển Hệ Thống bằng Jupyter Notebook!
Notebook này được tạo ra để giúp bạn **khởi chạy, kiểm soát, giám sát và phân tích** toàn bộ hệ thống Traffic Density Analysis một cách trực quan và tập trung nhất. Không còn phải mở hàng chục cửa sổ dòng lệnh (Terminal), tất cả giờ đây nằm gọn trong một giao diện duy nhất.

---

### 🏗️ Kiến trúc & Các thành phần hệ thống

1. **Backend (FastAPI)**: Cung cấp REST APIs lưu trữ dữ liệu nhận diện (`vehicle_detections`), tổng hợp mật độ (`traffic_aggregation`), và dự báo (`traffic_predictions`) lên MongoDB Atlas Cloud.
2. **Detection Engine (YOLOv9 + DeepSORT)**: Xử lý video luồng camera, phát hiện và theo dõi phương tiện, sau đó gửi sự kiện về Backend theo thời gian thực.
3. **Frontend Dashboard (React)**: Giao diện trực quan hóa thông tin mật độ giao thông, lịch sử ùn tắc thời gian thực.
4. **ML Service (TrafficPredictor)**: Dự báo mật độ giao thông ở chu kỳ tiếp theo.
5. **Integration Pipeline (system_runner)**: Bộ điều phối trung tâm chạy vòng lặp tuần hoàn để kiểm tra dữ liệu, tính toán ùn tắc.

---

### 🛠️ Cách sử dụng Notebook này
* **Bước 1**: Chạy cell **Kiểm tra môi trường** để xác minh thư viện và kết nối MongoDB.
* **Bước 2**: Định nghĩa **Bộ quản lý dịch vụ (System Manager)** để chuẩn bị các lệnh chạy ngầm.
* **Bước 3**: Khởi chạy hệ thống theo cách bạn muốn (Tích hợp 1-click hoặc Chạy từng Module độc lập để dễ gỡ lỗi).
* **Bước 4**: Giám sát dữ liệu & Phân tích trực quan thông qua biểu đồ thời gian thực được truy vấn trực tiếp từ MongoDB Atlas.

## 🔍 Bước 1: Kiểm tra Sức khỏe Môi trường & Kết nối Cơ sở dữ liệu
Chạy cell dưới đây để kiểm tra xem môi trường ảo (venv) đã được kích hoạt chưa, các thư viện cốt lõi có sẵn sàng không, GPU CUDA có khả dụng không và kết nối tới cụm dữ liệu MongoDB Atlas có thông suốt không.

In [ ]:
import os
import sys
import subprocess

print("==================================================================")
print("🔍 KIỂM TRA MÔI TRƯỜNG & KẾT NỐI HỆ THỐNG")
print("==================================================================")

# 1. Kiểm tra Virtual Environment
in_venv = sys.prefix != sys.base_prefix
print(f"🔹 Python Executable : {sys.executable}")
print(f"🔹 Đang chạy trong Virtual Env (venv): {'✅ ĐÃ KÍCH HOẠT' if in_venv else '⚠️ CHƯA KÍCH HOẠT (Có thể thiếu thư viện)'}")

# 2. Kiểm tra các thư viện cốt lõi
dependencies = [
    "fastapi", "uvicorn", "pymongo", "requests", "dotenv",
    "ultralytics", "torch", "pandas", "matplotlib", "seaborn", "psutil"
]
missing = []
print("\n🔹 Kiểm tra cài đặt thư viện:")
for dep in dependencies:
    try:
        if dep == "dotenv":
            __import__("dotenv")
        else:
            __import__(dep)
        print(f"   [✅] {dep:<15} - Sẵn sàng")
    except ImportError:
        print(f"   [❌] {dep:<15} - CHƯA CÀI ĐẶT!")
        missing.append(dep)

# 3. Kiểm tra GPU CUDA hỗ trợ YOLO
try:
    import torch
    cuda_avail = torch.cuda.is_available()
    if cuda_avail:
        print(f"\n🔹 PyTorch CUDA GPU  : ✅ KHẢ DỤNG | Thiết bị: {torch.cuda.get_device_name(0)}")
    else:
        print(f"\n🔹 PyTorch CUDA GPU  : ⚠️ KHÔNG KHẢ DỤNG | Chạy phát hiện xe bằng CPU (Sẽ chậm hơn)")
except ImportError:
    print("\n🔹 PyTorch CUDA GPU  : ❌ Không thể kiểm tra (Chưa cài đặt PyTorch)")

# 4. Kiểm tra file cấu hình .env và MongoDB Atlas
env_path = os.path.join(os.getcwd(), ".env")
print(f"\n🔹 Cấu hình .env      : {'✅ ĐÃ TÌM THẤY' if os.path.exists(env_path) else '❌ KHÔNG TÌM THẤY'}")

if os.path.exists(env_path):
    from dotenv import load_dotenv
    load_dotenv(env_path)
    db_url = os.getenv("DB_URL") or os.getenv("MONGODB_URI")
    if db_url:
        # Mask password for security
        masked_url = db_url
        if "@" in db_url:
            parts = db_url.split("@")
            prefix = parts[0].split(":")
            if len(prefix) > 2:
                prefix[2] = "******"
            masked_url = ":".join(prefix) + "@" + parts[1]
        print(f"🔹 Connection String  : {masked_url[:50]}...")
        
        try:
            from pymongo import MongoClient
            client = MongoClient(db_url, serverSelectionTimeoutMS=3000)
            client.admin.command('ping')
            print("   [✅] Kết nối MongoDB Atlas: THÀNH CÔNG (Ping OK)")
        except Exception as e:
            print(f"   [❌] Kết nối MongoDB Atlas: THẤT BẠI | Chi tiết: {e}")
    else:
        print("   [❌] Không tìm thấy khóa DB_URL hoặc MONGODB_URI trong file .env")

if missing:
    print("\n⚠️ CẢNH BÁO: Bạn thiếu một số thư viện quan trọng. Vui lòng cài đặt lại bằng lệnh:")
    print("   pip install -r requirements.txt")
print("==================================================================")

## 🛠️ Bước 2: Thiết lập Bộ Quản Lý Dịch Vụ (Traffic System Manager)
Chúng ta sẽ định nghĩa một class Python thông minh `TrafficSystemManager` giúp khởi tạo các luồng tiến trình chạy ngầm (subprocesses) cho từng dịch vụ và định hướng toàn bộ log của chúng ra thư mục `logs/` riêng biệt.

In [ ]:
import subprocess
import time
import os
import sys
import json

class TrafficSystemManager:
    def __init__(self):
        self.processes = {}
        self.project_root = os.getcwd()
        self.log_dir = os.path.join(self.project_root, "logs")
        os.makedirs(self.log_dir, exist_ok=True)
        
    def start_service(self, name, cmd, cwd=None, env_override=None):
        if self.is_running(name):
            print(f"⚠️ Dịch vụ '{name}' đang chạy sẵn (PID={self.processes[name]['process'].pid}).")
            return
            
        log_file_path = os.path.join(self.log_dir, f"{name}.log")
        # Mở file ghi đè (write mode)
        log_file = open(log_file_path, "w", encoding="utf-8")
        
        print(f"🚀 Khởi động dịch vụ '{name}'...")
        print(f"   📁 Thư mục: {cwd or self.project_root}")
        print(f"   📝 Ghi log: logs/{name}.log")
        
        # Chuẩn hóa command để dùng đúng python interpreter của venv
        actual_cmd = []
        for arg in cmd:
            if arg == "python":
                actual_cmd.append(sys.executable)
            else:
                actual_cmd.append(arg)
                
        # Setup env
        env = os.environ.copy()
        if env_override:
            env.update(env_override)
            
        # Chạy tiến trình ngầm cross-platform
        proc = subprocess.Popen(
            actual_cmd,
            cwd=cwd or self.project_root,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            creationflags=subprocess.CREATE_NEW_PROCESS_GROUP if os.name == 'nt' else 0
        )
        
        self.processes[name] = {
            "process": proc,
            "log_file": log_file,
            "log_path": log_file_path,
            "cmd": actual_cmd
        }
        
        time.sleep(2) # Đợi 2 giây xem có crash sớm không
        
        if proc.poll() is not None:
            print(f"❌ Dịch vụ '{name}' KHỞI ĐỘNG THẤT BẠI! Hãy kiểm tra logs/{name}.log.")
            log_file.close()
            self._show_last_log_lines(log_file_path)
        else:
            print(f"✅ Dịch vụ '{name}' KHỞI ĐỘNG THÀNH CÔNG (PID={proc.pid})")
            
    def is_running(self, name):
        if name in self.processes:
            proc_dict = self.processes[name]
            proc = proc_dict["process"]
            if proc.poll() is None:
                return True
        return False
        
    def stop_service(self, name):
        if not self.is_running(name):
            if name in self.processes:
                # Dọn dẹp file handles nếu đã die rồi
                try:
                    self.processes[name]["log_file"].close()
                except:
                    pass
                del self.processes[name]
            print(f"ℹ️ Dịch vụ '{name}' hiện không chạy.")
            return
            
        proc_dict = self.processes[name]
        proc = proc_dict["process"]
        print(f"🛑 Đang dừng dịch vụ '{name}' (PID={proc.pid})...")
        
        try:
            if os.name == 'nt':
                # Dùng taskkill để tắt sạch cả process con của nó trên Windows
                subprocess.run(["taskkill", "/F", "/T", "/PID", str(proc.pid)], capture_output=True)
            else:
                proc.terminate()
                try:
                    proc.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    proc.kill()
        except Exception as e:
            print(f"⚠️ Gặp lỗi khi cố dừng: {e}")
            try:
                proc.kill()
            except:
                pass
                
        try:
            proc_dict["log_file"].close()
        except:
            pass
            
        if name in self.processes:
            del self.processes[name]
        print(f"✅ Đã dừng hẳn dịch vụ '{name}'")
        
    def stop_all(self):
        print("🛑 TIẾN HÀNH DỪNG TOÀN BỘ CÁC DỊCH VỤ ĐANG CHẠY...")
        active_names = list(self.processes.keys())
        for name in active_names:
            self.stop_service(name)
        print("✅ Hoàn tất dừng toàn bộ hệ thống.")
        
    def show_status(self):
        print("==================================================================")
        print("📊 BẢNG TRẠNG THÁI DỊCH VỤ HỆ THỐNG")
        print("==================================================================")
        services = ["backend", "detection", "frontend", "integration"]
        for s in services:
            running = self.is_running(s)
            if running:
                pid = self.processes[s]["process"].pid
                print(f"🟢 {s:<12}: ĐANG CHẠY   | PID: {pid:<6} | Ghi log: logs/{s}.log")
            else:
                print(f"🔴 {s:<12}: ĐÃ DỪNG")
        print("==================================================================")
        
    def show_logs(self, name, lines=20):
        log_path = os.path.join(self.log_dir, f"{name}.log")
        if not os.path.exists(log_path):
            print(f"❌ Không tìm thấy file log: logs/{name}.log")
            return
            
        print(f"📖 Hiển thị {lines} dòng cuối cùng của log '{name}':")
        print("-" * 80)
        try:
            with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
                all_lines = f.readlines()
                for line in all_lines[-lines:]:
                    print(line.strip())
        except Exception as e:
            print(f"Lỗi đọc log: {e}")
        print("-" * 80)
        
    def _show_last_log_lines(self, log_path, lines=8):
        if os.path.exists(log_path):
            with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.readlines()
                print("   Dòng log cuối lỗi:")
                for line in content[-lines:]:
                    print(f"   | {line.strip()}")

# Khởi tạo bộ quản lý
manager = TrafficSystemManager()
print("✨ Bộ quản lý TrafficSystemManager đã sẵn sàng!")

## 🚀 Bước 3: Khởi Chạy Hệ Thống
Bạn có **2 lựa chọn linh hoạt** dưới đây để khởi chạy tùy vào nhu cầu phát triển hoặc vận hành:

### 🌟 LỰA CHỌN A: Khởi chạy tích hợp bằng 1 lệnh duy nhất (Một-cho-Tất-cả)
Khi chạy cách này, ta gọi file `system_runner.py`. Nó sẽ tự động khởi chạy và quản lý cả 3 luồng con: *Backend*, *Detection Engine*, và *Frontend (npm start)*. Rất phù hợp nếu bạn muốn chạy hệ thống nhanh chóng.

In [ ]:
# Khởi chạy tích hợp thông qua system_runner.py
# (Backend, Detection, Frontend sẽ tự động khởi chạy làm tiến trình con)
manager.start_service(
    name="integration",
    cmd=["python", "backend/system_runner.py"],
    env_override={"NO_SUBPROCESS": "0", "PIPELINE_INTERVAL": "5"}
)

### 🛠️ LỰA CHỌN B: Khởi chạy độc lập từng Module (Khuyên dùng khi lập trình/gỡ lỗi)
Chạy cách này giúp bạn phân rã luồng logs của từng thành phần ra các file log khác nhau (`logs/backend.log`, `logs/detection.log`, v.v.). Bạn có thể bật tắt riêng lẻ từng tiến trình dễ dàng.

In [ ]:
# Đảm bảo dừng luồng tích hợp trước để tránh xung đột cổng (port 8000)
if manager.is_running("integration"):
    manager.stop_service("integration")

print("🟢 Bắt đầu khởi chạy các Module độc lập...")

# 1. Khởi chạy Backend (FastAPI tại cổng 8000)
manager.start_service(
    name="backend",
    cmd=["python", "-m", "uvicorn", "backend.main:app", "--reload", "--host", "127.0.0.1", "--port", "8000"]
)

# Đợi backend lên ổn định trước khi chạy các phần khác kết nối tới nó
time.sleep(3)

# 2. Khởi chạy Detection Engine (YOLOv9 + DeepSORT)
manager.start_service(
    name="detection",
    cmd=["python", "-m", "detection.main"]
)

# 3. Khởi chạy Frontend React Dashboard (cổng 3000)
frontend_cmd = ["npm.cmd", "start"] if os.name == 'nt' else ["npm", "start"]
manager.start_service(
    name="frontend",
    cmd=frontend_cmd,
    cwd=os.path.join(os.getcwd(), "frontend")
)

# 4. Khởi chạy Integration Pipeline độc lập (chỉ làm nhiệm vụ điều phối)
manager.start_service(
    name="integration",
    cmd=["python", "backend/system_runner.py"],
    env_override={"NO_SUBPROCESS": "1", "PIPELINE_INTERVAL": "5"}
)

## 📈 Bước 4: Giám Sát & Xem Trạng Thái Hệ Thống

In [ ]:
# Kiểm tra bảng trạng thái hoạt động của các tiến trình
manager.show_status()

In [ ]:
# Ví dụ: Xem 15 dòng logs mới nhất của Backend
manager.show_logs("backend", lines=15)

In [ ]:
# Ví dụ: Xem 15 dòng logs mới nhất của bộ nhận diện YOLO (Detection)
manager.show_logs("detection", lines=15)

In [ ]:
# Ví dụ: Xem 15 dòng logs mới nhất của bộ điều phối (Integration)
manager.show_logs("integration", lines=15)

## 📊 Bước 5: Truy Vấn Dữ Liệu Thời Gian Thực & Vẽ Biểu Đồ Trực Quan
Với dữ liệu nhận diện liên tục được gửi về MongoDB Atlas, chúng ta hãy viết một đoạn mã Python kết nối trực tiếp đến Database và vẽ biểu đồ thể hiện **Số lượng xe đếm được** cùng **Chỉ số hàng đợi (Queue Proxy)** của luồng giao thông theo thời gian thực!

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from dotenv import load_dotenv
from datetime import datetime

# Load .env
load_dotenv()
db_url = os.getenv("DB_URL") or os.getenv("MONGODB_URI")
db_name = os.getenv("MONGODB_DB", "traffic_density")

if not db_url:
    print("❌ Không tìm thấy thông tin DB_URL trong file .env!")
else:
    try:
        client = MongoClient(db_url, serverSelectionTimeoutMS=3000)
        db = client[db_name]
        
        # 1. Thống kê số lượng record cơ bản
        det_count = db.vehicle_detections.count_documents({})
        agg_count = db.traffic_aggregation.count_documents({})
        pred_count = db.traffic_predictions.count_documents({})
        camera_count = db.cameras.count_documents({})
        
        print("==================================================================")
        print("📊 THỐNG KÊ KHO DỮ LIỆU MONGODB ATLAS")
        print("==================================================================")
        print(f"🔹 Tổng số Camera đã khai báo  : {camera_count:<5} thiết bị")
        print(f"🔹 Bản ghi nhận diện xe (raw)   : {det_count:<5} sự kiện")
        print(f"🔹 Bản ghi tổng hợp mật độ (agg) : {agg_count:<5} chu kỳ")
        print(f"🔹 Bản ghi dự báo mật độ (pred) : {pred_count:<5} chu kỳ")
        print("==================================================================\n")
        
        # 2. Truy vấn bảng tổng hợp mật độ mới nhất
        recent_aggs = list(db.traffic_aggregation.find().sort("timestamp", -1).limit(10))
        if recent_aggs:
            print("📈 TOP 10 BẢN GHI TỔNG HỢP MẬT ĐỘ MỚI NHẤT TRÊN ATLAS:")
            df_data = []
            for item in recent_aggs:
                ts = item.get("timestamp")
                if isinstance(ts, datetime):
                    ts_str = ts.strftime("%Y-%m-%d %H:%M:%S")
                else:
                    ts_str = str(ts)
                df_data.append({
                    "Thời gian": ts_str,
                    "Camera ID": item.get("camera_id"),
                    "Số xe đếm được": item.get("vehicle_count", 0),
                    "Lượng xe đi vào (Inbound)": item.get("inbound_count", 0),
                    "Hàng đợi ước lượng (Queue)": item.get("queue_proxy", 0),
                    "Mức ùn tắc": item.get("congestion_level", "N/A")
                })
            df = pd.DataFrame(df_data)
            display(df)
            
            # 3. Vẽ biểu đồ 25 bản ghi gần nhất
            chart_aggs = list(db.traffic_aggregation.find().sort("timestamp", -1).limit(25))
            chart_aggs.reverse() # Xếp theo dòng thời gian tăng dần
            
            time_labels = []
            for item in chart_aggs:
                t = item.get("timestamp")
                if isinstance(t, datetime):
                    time_labels.append(t.strftime("%H:%M:%S"))
                else:
                    time_labels.append(str(t)[-8:])
                    
            counts = [item.get("vehicle_count", 0) for item in chart_aggs]
            queues = [item.get("queue_proxy", 0) for item in chart_aggs]
            
            sns.set_theme(style="whitegrid")
            fig, ax1 = plt.subplots(figsize=(13, 6))
            
            color_blue = '#1e3c72'
            ax1.set_xlabel('Thời gian (H:M:S)', fontweight='bold', labelpad=10)
            ax1.set_ylabel('Số lượng phương tiện đếm được', color=color_blue, fontweight='bold')
            line1 = ax1.plot(time_labels, counts, color=color_blue, marker='o', linewidth=2.5, label='Số lượng xe')
            ax1.tick_params(axis='y', labelcolor=color_blue)
            ax1.set_xticklabels(time_labels, rotation=45, ha='right')
            
            # Trục tung bên phải biểu thị hàng đợi
            ax2 = ax1.twinx()
            color_orange = '#e65100'
            ax2.set_ylabel('Ước lượng Hàng đợi xe (Queue Proxy)', color=color_orange, fontweight='bold')
            line2 = ax2.plot(time_labels, queues, color=color_orange, marker='s', linestyle='--', linewidth=1.5, label='Chỉ số hàng đợi')
            ax2.tick_params(axis='y', labelcolor=color_orange)
            
            # Tổng hợp chú thích
            lines = line1 + line2
            labels = [l.get_label() for l in lines]
            ax1.legend(lines, labels, loc='upper left')
            
            plt.title('BIỂU ĐỒ BIẾN THIÊN MẬT ĐỘ GIAO THÔNG & CHỈ SỐ HÀNG ĐỢI', fontsize=14, fontweight='bold', pad=15, color='#1e3c72')
            fig.tight_layout()
            plt.show()
        else:
            print("⚠️ Không tìm thấy bản ghi tổng hợp nào trong collection 'traffic_aggregation'.")
            
    except Exception as e:
        print(f"❌ Lỗi kết nối truy vấn Database: {e}")

## 🛑 Bước 6: Tạm Dừng Hệ Thống
Sau khi hoàn tất phiên làm việc, bạn chạy cell dưới đây để dọn dẹp và dừng an toàn toàn bộ các luồng tiến trình chạy ngầm (tránh bị kẹt tiến trình chiếm dụng port).

In [ ]:
# Dừng toàn bộ các luồng dịch vụ
manager.stop_all()